# **1. Mount Google Drive**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **2. Mengimport Library**

In [2]:
import cv2
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops
import os

# **3. Menentukan Path Folder dan Properti Fitur**

In [3]:
# list properti yang akan diekstraksi
maturity_properties = ['hue_mean', 'saturation_mean', 'value_mean', 'hue_std', 'saturation_std', 'value_std', 'texture_contrast']

# path folder yang berisi gambar
folder_path = '/content/drive/MyDrive/KOMPUTER VISION 2318105/PISANG'
data = []

# **4. Loop Load Data file Citra dan Proses Ekstraksi Fitur**

In [4]:
# loop untuk setiap file citra di folder
for file_name in os.listdir(folder_path):
    if file_name.endswith('.jpg') or file_name.endswith('.jpeg') or file_name.endswith('.png') or file_name.endswith('.bmp'):
        row_data = []
        file_path = os.path.join(folder_path, file_name)
        row_data.append(file_name)

        # preprocessing gambar
        src = cv2.imread(file_path)
        src = cv2.resize(src, (300, 300))
        hsv = cv2.cvtColor(src, cv2.COLOR_BGR2HSV)
        h, s, v = cv2.split(hsv)

        # ekstraksi properti warna
        hue_mean = np.mean(h)
        saturation_mean = np.mean(s)
        value_mean = np.mean(v)
        hue_std = np.std(h)
        saturation_std = np.std(s)
        value_std = np.std(v)

        # ekstraksi properti bentuk
        contours, hierarchy = cv2.findContours(cv2.Canny(src, 100, 200), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        cnt = max(contours, key=cv2.contourArea)
        perimeter = cv2.arcLength(cnt, True)
        area = cv2.contourArea(cnt)
        circularity = 4 * np.pi * (area / perimeter ** 2)
        aspect_ratio = float(src.shape[0]) / src.shape[1]
        extent = area / (src.shape[0] * src.shape[1])
        solidity = area / cv2.contourArea(cv2.convexHull(cnt))

        # hitung kontras tekstur
        g = graycomatrix(v, distances=[1], angles=[0], symmetric=True, normed=True)
        contrast = graycoprops(g, 'contrast')
        # tambahkan ke data
        row_data.extend([hue_mean, saturation_mean, value_mean, hue_std, saturation_std, value_std, contrast])
        data.append(row_data)

# **5. Export Data Hasil Ekstraksi Fitur ke File CSV**

In [5]:
# export ke file CSV
df = pd.DataFrame(data, columns=['file_name'] + maturity_properties)
file_path = os.path.join(folder_path, '2318105EkstraksFiturPisang.csv')
df.to_csv(file_path, index=False)
print("File CSV berhasil disimpan!")
print(df)

File CSV berhasil disimpan!
                        file_name   hue_mean  saturation_mean  value_mean  \
0  pisang_03_setengah_matang.jpeg  13.588500        76.169533  229.990711   
1   pisang_04_terlalu_matang.jpeg  31.408189        46.794133  218.359256   
2       pisang_05_mentah.jpg.jpeg   9.377922        20.435500  244.234756   
3           pisang_01_matang.jpeg   9.024722        53.488589  237.053656   
4           pisang_06_mentah.jpeg  16.582211        35.345289  238.458856   
5           pisang_02_matang.jpeg  36.581389       100.912233  182.336044   
6           pisang_07_matang.jpeg  11.106211        69.877522  244.256711   
7   pisang_08_terlalu_matang.jpeg   6.650711        44.372656  222.536533   
8   pisang_09_terlalu_matang.jpeg  34.314400        21.971256  220.202278   
9           pisang_10_matang.jpeg  10.263567        66.040722  237.201478   

     hue_std  saturation_std  value_std        texture_contrast  
0  17.104206       97.635615  43.149829   [[95.32095875139